# 15 - Ten hospitals (organism_community), validation-selected protocol + fingerprint controls

Same split and budget as notebook 12 (10 hospitals, 5 seeds, rounds=8, local_epochs=4,
local_only_epochs=32, hidden=128), with two changes:

1. **Validation-selected protocol** (see notebook 14 / `federated/selection.py`): the (round, cutoff)
   is chosen on validation macro-F1 and the test score is read once there. Local-only gets the same rule.
   Old-protocol numbers (`fedavg_best`, ...) are still stored for comparison.
2. **Two controls for the topology arm** (eight arms in total):
   - `topo_shuffled` - every hospital is handed *another* hospital's fingerprint (a seeded derangement,
     different per seed). Same weight spread as real topo, attached to the wrong hospitals. **If real topo
     does not beat this, the fingerprint carries no usable signal.**
   - `invsize` - weights proportional to 1/n_train (small hospitals count more), no graph information.
     Tests whether topo is just "do not size-weight".

**Splitting the work across two accounts / sessions:** set `RUN_ARMS` in cell 4 to a subset (for example
`['fedavg','fedprox','fedmedian','fedtrimmedavg']` on one and `['uniform','topo','topo_shuffled','invsize']`
on the other). Each writes its own `results/amr_org10_val_<TAG>.json`; cell 8 merges them. Paired
comparisons are cleanest when everything ran on the same kind of hardware.

In [ ]:
# 1) Environment: Colab (clone + install + mount Drive) or local (.venv)
import os, sys
ON_COLAB = 'google.colab' in sys.modules
BRANCH = 'feat/federated-baselines'
if ON_COLAB:
    !git clone -b main https://github.com/RawEgg6/Capstone-amr-fed.git 2>/dev/null || (cd Capstone-amr-fed && git fetch && git checkout main && git pull)
    !cd /content/Capstone-amr-fed && git fetch origin {BRANCH} && git checkout {BRANCH} && git pull origin {BRANCH}
    !cd /content/Capstone-amr-fed && git log -1 --oneline
    !pip install -q torch_geometric 'flwr[simulation]' networkx
    from google.colab import drive
    drive.mount('/content/drive')
    os.environ['ARMD_DIR'] = '/content/drive/MyDrive/ARMD'   # edit to your folder if different
else:
    os.environ.setdefault('ARMD_DIR', '/Users/vikramadityasharma/Library/CloudStorage/GoogleDrive-vikramaditya.raunak@gmail.com/.shortcut-targets-by-id/1uVx5PmldPyJBMtnB9Lkl9N1rNBQrmJt-/ARMD')
import torch, ray, flwr, torch_geometric
print('colab:', ON_COLAB, '| python', sys.version.split()[0], '| torch', torch.__version__, '| flwr', flwr.__version__,
      '| ray', ray.__version__, '| device:', 'cuda' if torch.cuda.is_available() else 'cpu')
print('ARMD_DIR:', os.environ['ARMD_DIR'])

In [ ]:
# 2) Sanity: data resolves + imports (selection + derangement must exist = right branch)
from pathlib import Path
if ON_COLAB:
    SRC = Path('/content/Capstone-amr-fed/src')
else:
    SRC = next(p / 'src' for p in [Path.cwd(), *Path.cwd().parents] if (p / 'src' / 'amr_fed').is_dir())
sys.path.insert(0, str(SRC))
from amr_fed import config
D = Path(config.DATA_DIR)
print('DATA_DIR:', D, '| exists:', D.exists())
assert D.exists(), 'ARMD_DIR is wrong - fix cell 1 and re-run.'
from amr_fed.federated import selection
from amr_fed.topology import derangement, compute_fingerprint, aggregation_weights
from amr_fed.federated.run import run_multiseed
print('imports OK | example derangement(10, 42):', derangement(10, 42))

In [ ]:
# 3) Load cohort once + smoke (organism_community is deterministic -> same partition for every seed)
import sys as _s
for _m in [m for m in list(_s.modules) if m.startswith('amr_fed')]:
    del _s.modules[_m]
from amr_fed.data_loader import load_cohort_frame, PK
from amr_fed.partition import organism_community, partition_summary
df = load_cohort_frame()
print('rows:', f'{len(df):,}')
SEEDS = (42, 43, 44, 45, 46)
assign = organism_community(df, n_clients=10)
g, niid = partition_summary(df, assign)
print(g)
assert assign.nunique() == 10, 'did not produce 10 hospitals - stop and report'
assert assign.index.is_unique and len(assign) == df[PK].nunique(), 'not 1:1'
print('SMOKE PASS')

In [ ]:
# 4) Arms. Edit RUN_ARMS / TAG to split work across accounts. Checkpoints after every arm and resumes.
import os, ray, json, shutil
os.environ.pop('AMR_RNG_PROBE', None)
import sys as _s
for _m in [m for m in list(_s.modules) if m.startswith('amr_fed')]:
    del _s.modules[_m]
from amr_fed.federated.run import run_multiseed
from amr_fed.partition import organism_community

ALL_ARMS = [  # (key, label, extra kwargs)
    ('fedavg',        'org10/FedAvg',        dict(strategy='fedavg')),
    ('fedprox',       'org10/FedProx',       dict(strategy='fedprox')),
    ('fedmedian',     'org10/FedMedian',     dict(strategy='fedmedian')),
    ('fedtrimmedavg', 'org10/FedTrimmedAvg', dict(strategy='fedtrimmedavg')),
    ('uniform',       'org10/UNIFORM (T=inf control)', dict(strategy='topology', topo_mode='distinctiveness', topo_temperature=float('inf'))),
    ('topo',          'org10/TOPO (T=1)',    dict(strategy='topology', topo_mode='distinctiveness', topo_temperature=1.0)),
    ('topo_shuffled', 'org10/TOPO-SHUFFLED (control)', dict(strategy='topology', topo_mode='distinctiveness', topo_temperature=1.0, topo_shuffle=True)),
    ('invsize',       'org10/INV-SIZE (control)',      dict(strategy='topology', topo_mode='invsize')),
]
RUN_ARMS = [k for k, _, _ in ALL_ARMS]        # <- subset here to split across accounts
TAG = 'all'                                    # <- file suffix: amr_org10_val_<TAG>.json
KW = dict(seeds=SEEDS, df=df, partition_fn=lambda d, s: organism_community(d, n_clients=10, seed=s),
          n_clients=10, rounds=8, local_epochs=4, local_only_epochs=32, hidden=128)
RESULTS = SRC.parent / 'results' / f'amr_org10_val_{TAG}.json'
RESULTS.parent.mkdir(exist_ok=True)
MIRROR = Path(f'/content/drive/MyDrive/amr_org10_val_{TAG}.json') if ON_COLAB else None
src = RESULTS if RESULTS.exists() else (MIRROR if MIRROR and MIRROR.exists() else None)
res = json.load(open(src)) if src else {}
for key, label, extra in ALL_ARMS:
    if key not in RUN_ARMS:
        continue
    if key in res:
        print(f'skip {key}: already saved')
        continue
    ray.shutdown()
    # pooled is computed once, in the fedavg arm only
    res[key] = run_multiseed(label=label, compute_pooled=(key == 'fedavg'), **KW, **extra)
    json.dump(res, open(RESULTS, 'w'), default=str)
    if MIRROR: shutil.copy(RESULTS, MIRROR)
    print(f'checkpoint saved after {key}')

In [ ]:
# 5) Summary, validation-selected protocol (mean +/- std over seeds; test read once at the val choice)
import numpy as np
names = {'fedavg':'FedAvg','fedprox':'FedProx','fedmedian':'FedMedian','fedtrimmedavg':'FedTrimmedAvg','uniform':'Uniform',
         'topo':'Topo','topo_shuffled':'Topo-shuffled','invsize':'Inv-size'}
keys = [k for k, _, _ in ALL_ARMS if k in res]
def col(k, key): return np.array([r[key] for r in res[k]['runs']], dtype=float)
def f(x): return f'{np.nanmean(x):.4f} +/- {np.nanstd(x):.4f}'
ref = keys[0]
lo = col(ref, 'val_local')
print(f"{'method':<15}{'val-sel F1':<18}{'@0.5, val round':<18}{'final, val cutoff':<19}{'worst hospital':<18}{'AUROC':<18}{'val round / seed'}")
print(f"{'local-only':<15}{f(lo):<18}{'-':<18}{'-':<19}{f(col(ref,'val_worst_local')):<18}{f(col(ref,'val_local_auroc')):<18}")
for k in keys:
    print(f"{names[k]:<15}{f(col(k,'val_fed')):<18}{f(col(k,'val_fed_at05')):<18}{f(col(k,'val_fed_final')):<19}{f(col(k,'val_worst_fed')):<18}{f(col(k,'val_fed_auroc')):<18}{[r['val_fed_round'] for r in res[k]['runs']]}")
print('\nfederated - local-only (val-selected, per seed):')
for k in keys:
    d = col(k, 'val_fed') - lo
    print(f'  {names[k]:<15}{d.mean():+.4f}  {np.round(d, 4)}')
print('\nOLD protocol (best round on TEST, fixed 0.5), same run, for reference:')
for k in keys: print(f"  {names[k]:<15}{f(col(k,'fedavg_best'))}  (final {f(col(k,'fedavg_final'))})")

In [ ]:
# 6) THE decisive comparison: does the real fingerprint beat its controls? (paired by seed)
from scipy import stats
def paired(a, b, metric):
    d = col(a, metric) - col(b, metric)
    se = d.std(ddof=1) / np.sqrt(len(d)); t = stats.t.ppf(.975, len(d) - 1)
    return d, d.mean(), d.mean() - t * se, d.mean() + t * se, stats.ttest_1samp(d, 0).pvalue
for metric, label in [('val_fed', 'val-selected F1'), ('val_fed_final', 'final round, val cutoff'), ('val_worst_fed', 'worst hospital')]:
    for a, b in [('topo','topo_shuffled'), ('topo','invsize'), ('topo','uniform'), ('uniform','fedavg'), ('topo','fedmedian')]:
        if a in res and b in res:
            d, m, lo_ci, hi_ci, p = paired(a, b, metric)
            print(f'{label:<26} {names[a]:>8} - {names[b]:<14} {m:+.4f}  95% CI [{lo_ci:+.4f}, {hi_ci:+.4f}]  p={p:.2f}  per seed {np.round(d, 4)}')
    print()

In [ ]:
# 7) The weights each arm actually used, recomputed from fingerprints (partition is fixed)
ids = assign.value_counts().sort_values(ascending=False).index.tolist()   # hospital ids, largest first
subs = {c: df[df[PK].map(assign) == c] for c in ids}
fps = {c: compute_fingerprint(subs[c]) for c in ids}
real = dict(zip(ids, aggregation_weights([fps[c] for c in ids], temperature=1.0, mode='distinctiveness')))
sizes = {c: subs[c][PK].nunique() for c in ids}
inv = {c: 1 / sizes[c] for c in ids}; z = sum(inv.values()); inv = {c: v / z for c, v in inv.items()}   # approx (patients, not train triples)
print(f"{'hospital':<9}{'patients':>9}{'size share':>11}{'uniform':>9}{'real topo':>10}{'inv-size':>10}")
for c in ids:
    print(f'{c:<9}{sizes[c]:>9}{sizes[c]/sum(sizes.values()):>11.3f}{0.1:>9.3f}{real[c]:>10.3f}{inv[c]:>10.3f}')
if 'topo_shuffled' in res:
    print('\nshuffled arm: hospital c gets the weight of hospital perm[c] (perm differs per seed)')
    for r, s in zip(res['topo_shuffled']['runs'], SEEDS):
        perm = r['topo_shuffle_perm']
        print(f'  seed {s}: perm {perm} -> heaviest hospital under shuffle: {max(range(10), key=lambda c: real[perm[c]])} (real topo: {max(real, key=real.get)})')

In [ ]:
# 8) (Optional) Merge JSONs from several accounts / sessions into one file, then re-run cells 5-7
import glob
parts = sorted(glob.glob(str(SRC.parent / 'results' / 'amr_org10_val_*.json')))
if ON_COLAB: parts += sorted(glob.glob('/content/drive/MyDrive/amr_org10_val_*.json'))
merged = {}
for p in parts:
    if p.endswith('amr_org10_val_merged.json'): continue
    for k, v in json.load(open(p)).items():
        merged.setdefault(k, v)
print('files:', parts, '| arms:', list(merged))
json.dump(merged, open(SRC.parent / 'results' / 'amr_org10_val_merged.json', 'w'), default=str)
# to analyse the merged set: res = merged ; then re-run cells 5-7